In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import binomtest
from tqdm import tqdm


def analyze_sv_gen_correlation(data, sample_num=None, verbose=False):
    """
    Analyze correlation between SV/VS order and GEN-N/N-GEN order.
    
    Hypothesis:
    - VS order correlates with N-GEN (PSSD-PSSR)
    - SV order correlates with GEN-N (PSSR-PSSD)
    """
    
    data = data.copy()
    
    # GB130: SV/VS order
    # '1' = SV
    # '2' = VS
    data['SV_Order'] = 'Unknown'
    data.loc[data['GB130'] == '1', 'SV_Order'] = 'SV'
    data.loc[data['GB130'] == '2', 'SV_Order'] = 'VS'
    
    # GB065: Possessor order (using GEN-N terminology)
    # '1' = PSSR-PSSD = GEN-N
    # '2' = PSSD-PSSR = N-GEN
    data['GEN_Order'] = 'Unknown'
    data.loc[data['GB065'] == '1', 'GEN_Order'] = 'GEN-N'
    data.loc[data['GB065'] == '2', 'GEN_Order'] = 'N-GEN'
    
    # Filter valid data
    valid_data = data[(data['SV_Order'] != 'Unknown') & (data['GEN_Order'] != 'Unknown')]
    
    if len(valid_data) < 10:
        return None
    
    # Get SV and VS subsets
    sv_langs = valid_data[valid_data['SV_Order'] == 'SV']
    vs_langs = valid_data[valid_data['SV_Order'] == 'VS']
    
    # Count expected patterns
    # SV with GEN-N (expected)
    sv_gen_n = sum(sv_langs['GEN_Order'] == 'GEN-N')
    sv_total = len(sv_langs)
    
    # VS with N-GEN (expected)
    vs_n_gen = sum(vs_langs['GEN_Order'] == 'N-GEN')
    vs_total = len(vs_langs)
    
    # Calculate proportions
    sv_gen_n_prop = sv_gen_n / sv_total if sv_total > 0 else np.nan
    vs_n_gen_prop = vs_n_gen / vs_total if vs_total > 0 else np.nan
    
    # Binomial tests (null hypothesis: p = 0.5)
    sv_binom = binomtest(sv_gen_n, n=sv_total, p=0.5, alternative='greater') if sv_total >= 5 else None
    vs_binom = binomtest(vs_n_gen, n=vs_total, p=0.5, alternative='greater') if vs_total >= 5 else None
    
    sv_p = sv_binom.pvalue if sv_binom else np.nan
    vs_p = vs_binom.pvalue if vs_binom else np.nan
    
    if verbose:
        print(f"\nSample {sample_num}:")
        print(f"Total valid languages: {len(valid_data)}")
        print(f"SV languages: {sv_total}, SV→GEN-N: {sv_gen_n} ({sv_gen_n_prop:.1%}), p={sv_p:.4f}")
        print(f"VS languages: {vs_total}, VS→N-GEN: {vs_n_gen} ({vs_n_gen_prop:.1%}), p={vs_p:.4f}")
        print("-" * 70)
    
    return {
        'sample': sample_num,
        'n_languages': len(valid_data),
        'sv_total': sv_total,
        'vs_total': vs_total,
        'sv_gen_n': sv_gen_n,
        'vs_n_gen': vs_n_gen,
        'sv_gen_n_prop': sv_gen_n_prop,
        'vs_n_gen_prop': vs_n_gen_prop,
        'sv_p_value': sv_p,
        'vs_p_value': vs_p,
        'sv_significant': sv_p < 0.05 if not np.isnan(sv_p) else False,
        'vs_significant': vs_p < 0.05 if not np.isnan(vs_p) else False
    }


def analyze_sv_gen_head_marking(data, sample_num=None, verbose=False):
    """Same analysis for head-marking languages only"""
    
    if 'GB431' not in data.columns or 'GB433' not in data.columns:
        return None
    
    head_marking = data[(data['GB431'] == '1') | (data['GB433'] == '1')].copy()
    
    if len(head_marking) < 10:
        return None
    
    return analyze_sv_gen_correlation(head_marking, sample_num, verbose)


def create_stratified_sample(data, max_total=120, langs_per_area=20):
    """
    Create stratified sample following Hammarström & Donohue (2014) methodology.
    
    Parameters:
    - max_total: Maximum 120 languages total
    - langs_per_area: 20 language families per macroarea
    - All languages from different families
    """
    sampled_data = pd.DataFrame()
    
    # Get macroareas
    macroareas = data['Macroarea'].dropna().unique()
    
    for area in macroareas:
        area_data = data[data['Macroarea'] == area].copy()
        if area_data.empty:
            continue
        
        # Get unique families
        families = area_data['Family'].unique()
        families = families[~pd.isna(families)]
        
        if len(families) == 0:
            continue
        
        # Sample up to langs_per_area families
        num_to_sample = min(len(families), langs_per_area)
        selected_families = np.random.choice(families, size=num_to_sample, replace=False)
        
        # Sample one language per family
        for family in selected_families:
            family_data = area_data[area_data['Family'] == family]
            if not family_data.empty:
                sampled_lang = family_data.sample(n=1)
                sampled_data = pd.concat([sampled_data, sampled_lang])
    
    # Ensure we don't exceed max_total
    if len(sampled_data) > max_total:
        sampled_data = sampled_data.sample(n=max_total)
    
    return sampled_data


def visualize_results(all_langs_results, head_marking_results):
    """Create comprehensive visualizations"""
    
    all_df = pd.DataFrame([r for r in all_langs_results if r is not None])
    hm_df = pd.DataFrame([r for r in head_marking_results if r is not None])
    
    fig, axes = plt.subplots(3, 2, figsize=(14, 14))
    
    # 1. SV → GEN-N proportion distribution
    ax = axes[0, 0]
    if len(all_df) > 0:
        sns.histplot(all_df['sv_gen_n_prop'].dropna() * 100, kde=True, bins=20, ax=ax, color='steelblue')
        mean_val = all_df['sv_gen_n_prop'].mean() * 100
        ax.axvline(mean_val, color='red', linestyle='--', linewidth=2,
                   label=f'Mean: {mean_val:.1f}%')
        ax.axvline(50, color='black', linestyle=':', linewidth=2, label='Chance (50%)')
        ax.set_xlabel('% SV languages with GEN-N', fontsize=11)
        ax.set_ylabel('Count', fontsize=11)
        ax.set_title('All Languages: SV → GEN-N Distribution', fontsize=12, fontweight='bold')
        ax.legend()
        ax.set_xlim(0, 100)
    
    # 2. VS → N-GEN proportion distribution
    ax = axes[0, 1]
    if len(all_df) > 0:
        sns.histplot(all_df['vs_n_gen_prop'].dropna() * 100, kde=True, bins=20, ax=ax, color='coral')
        mean_val = all_df['vs_n_gen_prop'].mean() * 100
        ax.axvline(mean_val, color='red', linestyle='--', linewidth=2,
                   label=f'Mean: {mean_val:.1f}%')
        ax.axvline(50, color='black', linestyle=':', linewidth=2, label='Chance (50%)')
        ax.set_xlabel('% VS languages with N-GEN', fontsize=11)
        ax.set_ylabel('Count', fontsize=11)
        ax.set_title('All Languages: VS → N-GEN Distribution', fontsize=12, fontweight='bold')
        ax.legend()
        ax.set_xlim(0, 100)
    
    # 3. P-value distribution - SV
    ax = axes[1, 0]
    if len(all_df) > 0:
        p_vals = all_df['sv_p_value'].dropna()
        sns.histplot(p_vals, bins=20, ax=ax, color='steelblue')
        ax.axvline(0.05, color='red', linestyle='--', linewidth=2, label='α = 0.05')
        sig_count = sum(p_vals < 0.05)
        ax.set_xlabel('P-value', fontsize=11)
        ax.set_ylabel('Count', fontsize=11)
        ax.set_title(f'SV P-values ({sig_count}/{len(p_vals)} significant)', 
                     fontsize=12, fontweight='bold')
        ax.legend()
    
    # 4. P-value distribution - VS
    ax = axes[1, 1]
    if len(all_df) > 0:
        p_vals = all_df['vs_p_value'].dropna()
        sns.histplot(p_vals, bins=20, ax=ax, color='coral')
        ax.axvline(0.05, color='red', linestyle='--', linewidth=2, label='α = 0.05')
        sig_count = sum(p_vals < 0.05)
        ax.set_xlabel('P-value', fontsize=11)
        ax.set_ylabel('Count', fontsize=11)
        ax.set_title(f'VS P-values ({sig_count}/{len(p_vals)} significant)', 
                     fontsize=12, fontweight='bold')
        ax.legend()
    
    # 5. Comparison: All vs Head-marking (SV)
    ax = axes[2, 0]
    if len(all_df) > 0:
        data_to_plot = [all_df['sv_gen_n_prop'].dropna() * 100]
        labels = ['All Languages']
        colors = ['steelblue']
        
        if len(hm_df) > 0:
            data_to_plot.append(hm_df['sv_gen_n_prop'].dropna() * 100)
            labels.append('Head-Marking')
            colors.append('coral')
        
        bp = ax.boxplot(data_to_plot, labels=labels, patch_artist=True)
        for patch, color in zip(bp['boxes'], colors):
            patch.set_facecolor(color)
        
        ax.axhline(50, color='black', linestyle=':', linewidth=2, label='Chance')
        ax.set_ylabel('% with GEN-N', fontsize=11)
        ax.set_title('SV → GEN-N Comparison', fontsize=12, fontweight='bold')
        ax.legend()
        ax.set_ylim(0, 100)
    
    # 6. Comparison: All vs Head-marking (VS)
    ax = axes[2, 1]
    if len(all_df) > 0:
        data_to_plot = [all_df['vs_n_gen_prop'].dropna() * 100]
        labels = ['All Languages']
        colors = ['steelblue']
        
        if len(hm_df) > 0:
            data_to_plot.append(hm_df['vs_n_gen_prop'].dropna() * 100)
            labels.append('Head-Marking')
            colors.append('coral')
        
        bp = ax.boxplot(data_to_plot, labels=labels, patch_artist=True)
        for patch, color in zip(bp['boxes'], colors):
            patch.set_facecolor(color)
        
        ax.axhline(50, color='black', linestyle=':', linewidth=2, label='Chance')
        ax.set_ylabel('% with N-GEN', fontsize=11)
        ax.set_title('VS → N-GEN Comparison', fontsize=12, fontweight='bold')
        ax.legend()
        ax.set_ylim(0, 100)
    
    plt.tight_layout()
    plt.savefig('sv_vs_gen_correlation_stratified.png', dpi=300, bbox_inches='tight')
    print("\nVisualization saved to: sv_vs_gen_correlation_stratified.png")
    plt.close()


def main():
    # Set random seed for reproducibility
    np.random.seed(42)
    
    # Load data
    print("Loading data...")
    try:
        grambank = pd.read_csv('grambank_sane_format.csv')
        languages = pd.read_csv('languages1.csv')
    except FileNotFoundError as e:
        print(f"Error: {e}")
        print("Please ensure CSV files are in the current directory")
        return
    
    # Add metadata
    metadata = {}
    for _, row in languages.iterrows():
        if pd.notna(row['Name']):
            metadata[row['Name']] = {
                'macroarea': row['Macroarea'],
                'family': row['Family_name']
            }
    
    grambank['Macroarea'] = grambank['Language'].map(lambda x: metadata.get(x, {}).get('macroarea'))
    grambank['Family'] = grambank['Language'].map(lambda x: metadata.get(x, {}).get('family'))
    
    # Convert columns to string
    required_columns = ['GB130', 'GB065', 'GB431', 'GB433']
    for col in required_columns:
        if col in grambank.columns:
            grambank[col] = grambank[col].astype(str)
    
    print(f"Data loaded: {len(grambank)} languages")
    
    # Run stratified analysis
    print("\n" + "="*70)
    print("SV/VS vs GEN-N/N-GEN CORRELATION ANALYSIS")
    print("Following Hammarström & Donohue (2014) methodology")
    print("="*70)
    print("\nHypotheses:")
    print("  H1: SV order correlates with GEN-N (PSSR-PSSD)")
    print("  H2: VS order correlates with N-GEN (PSSD-PSSR)")
    print("\nMethod: 300 stratified samples, max 120 languages each")
    print("        20 families per macroarea, binomial tests (p=0.5)")
    
    n_samples = 300
    all_langs_results = []
    head_marking_results = []
    
    print("\nProcessing samples...")
    for i in tqdm(range(n_samples), desc="Stratified sampling"):
        sample = create_stratified_sample(grambank, max_total=120, langs_per_area=20)
        
        # All languages
        result_all = analyze_sv_gen_correlation(sample, sample_num=i+1, verbose=False)
        all_langs_results.append(result_all)
        
        # Head-marking
        result_hm = analyze_sv_gen_head_marking(sample, sample_num=i+1, verbose=False)
        head_marking_results.append(result_hm)
    
    # Summary statistics - All languages
    print("\n" + "="*70)
    print("RESULTS - ALL LANGUAGES")
    print("="*70)
    
    valid_all = [r for r in all_langs_results if r is not None]
    
    if len(valid_all) > 0:
        # SV → GEN-N
        sv_props = [r['sv_gen_n_prop'] * 100 for r in valid_all if not np.isnan(r['sv_gen_n_prop'])]
        sv_ps = [r['sv_p_value'] for r in valid_all if not np.isnan(r['sv_p_value'])]
        sv_sig = sum(1 for r in valid_all if r['sv_significant'])
        
        # VS → N-GEN
        vs_props = [r['vs_n_gen_prop'] * 100 for r in valid_all if not np.isnan(r['vs_n_gen_prop'])]
        vs_ps = [r['vs_p_value'] for r in valid_all if not np.isnan(r['vs_p_value'])]
        vs_sig = sum(1 for r in valid_all if r['vs_significant'])
        
        print(f"\nValid samples: {len(valid_all)}/{n_samples}")
        
        print(f"\nH1: SV → GEN-N (PSSR-PSSD)")
        print(f"  Mean: {np.mean(sv_props):.1f}% (±{np.std(sv_props):.1f}% SD)")
        print(f"  Median: {np.median(sv_props):.1f}%")
        print(f"  Range: [{np.min(sv_props):.1f}%, {np.max(sv_props):.1f}%]")
        print(f"  Significant samples: {sv_sig}/{len(sv_ps)} ({sv_sig/len(sv_ps):.1%})")
        
        if np.mean(sv_props) > 50:
            print(f"  Result: ✓ SUPPORTED (mean > 50%)")
        else:
            print(f"  Result: ✗ NOT SUPPORTED (mean < 50%)")
        
        print(f"\nH2: VS → N-GEN (PSSD-PSSR)")
        print(f"  Mean: {np.mean(vs_props):.1f}% (±{np.std(vs_props):.1f}% SD)")
        print(f"  Median: {np.median(vs_props):.1f}%")
        print(f"  Range: [{np.min(vs_props):.1f}%, {np.max(vs_props):.1f}%]")
        print(f"  Significant samples: {vs_sig}/{len(vs_ps)} ({vs_sig/len(vs_ps):.1%})")
        
        if np.mean(vs_props) > 50:
            print(f"  Result: ✓ SUPPORTED (mean > 50%)")
        else:
            print(f"  Result: ✗ NOT SUPPORTED (mean < 50%)")
    
    # Summary statistics - Head-marking
    print("\n" + "="*70)
    print("RESULTS - HEAD-MARKING LANGUAGES")
    print("="*70)
    
    valid_hm = [r for r in head_marking_results if r is not None]
    
    if len(valid_hm) > 0:
        # SV → GEN-N
        sv_props_hm = [r['sv_gen_n_prop'] * 100 for r in valid_hm if not np.isnan(r['sv_gen_n_prop'])]
        sv_sig_hm = sum(1 for r in valid_hm if r['sv_significant'])
        
        # VS → N-GEN
        vs_props_hm = [r['vs_n_gen_prop'] * 100 for r in valid_hm if not np.isnan(r['vs_n_gen_prop'])]
        vs_sig_hm = sum(1 for r in valid_hm if r['vs_significant'])
        
        print(f"\nValid samples: {len(valid_hm)}/{n_samples}")
        
        print(f"\nH1: SV → GEN-N (PSSR-PSSD)")
        print(f"  Mean: {np.mean(sv_props_hm):.1f}% (±{np.std(sv_props_hm):.1f}% SD)")
        print(f"  Significant samples: {sv_sig_hm}/{len(sv_props_hm)} ({sv_sig_hm/len(sv_props_hm):.1%})")
        
        print(f"\nH2: VS → N-GEN (PSSD-PSSR)")
        print(f"  Mean: {np.mean(vs_props_hm):.1f}% (±{np.std(vs_props_hm):.1f}% SD)")
        print(f"  Significant samples: {vs_sig_hm}/{len(vs_props_hm)} ({vs_sig_hm/len(vs_props_hm):.1%})")
        
        # Comparison
        print("\n" + "="*70)
        print("COMPARISON: ALL vs HEAD-MARKING")
        print("="*70)
        print(f"\nSV → GEN-N:")
        print(f"  All languages:  {np.mean(sv_props):.1f}%")
        print(f"  Head-marking:   {np.mean(sv_props_hm):.1f}%")
        print(f"  Difference:     {abs(np.mean(sv_props) - np.mean(sv_props_hm)):.1f} pp")
        
        print(f"\nVS → N-GEN:")
        print(f"  All languages:  {np.mean(vs_props):.1f}%")
        print(f"  Head-marking:   {np.mean(vs_props_hm):.1f}%")
        print(f"  Difference:     {abs(np.mean(vs_props) - np.mean(vs_props_hm)):.1f} pp")
    else:
        print("\nInsufficient head-marking samples for analysis")
    
    # Create visualizations
    print("\n" + "="*70)
    print("CREATING VISUALIZATIONS")
    print("="*70)
    visualize_results(all_langs_results, head_marking_results)
    
    # Save results
    all_df = pd.DataFrame([r for r in all_langs_results if r is not None])
    all_df.to_csv('sv_vs_gen_all_languages.csv', index=False)
    print("Detailed results (all) saved to: sv_vs_gen_all_languages.csv")
    
    if len(valid_hm) > 0:
        hm_df = pd.DataFrame([r for r in head_marking_results if r is not None])
        hm_df.to_csv('sv_vs_gen_head_marking.csv', index=False)
        print("Detailed results (head-marking) saved to: sv_vs_gen_head_marking.csv")
    
    print("\n" + "="*70)
    print("ANALYSIS COMPLETE!")
    print("="*70)


if __name__ == "__main__":
    main()

Loading data...
Data loaded: 2467 languages

SV/VS vs GEN-N/N-GEN CORRELATION ANALYSIS
Following Hammarström & Donohue (2014) methodology

Hypotheses:
  H1: SV order correlates with GEN-N (PSSR-PSSD)
  H2: VS order correlates with N-GEN (PSSD-PSSR)

Method: 300 stratified samples, max 120 languages each
        20 families per macroarea, binomial tests (p=0.5)

Processing samples...


Stratified sampling: 100%|████████████████████| 300/300 [00:09<00:00, 32.19it/s]
/var/folders/jk/cgmd03wn0g1b85mr8y_w4__c0000gn/T/ipykernel_11337/1557348027.py:219: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(data_to_plot, labels=labels, patch_artist=True)
/var/folders/jk/cgmd03wn0g1b85mr8y_w4__c0000gn/T/ipykernel_11337/1557348027.py:241: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(data_to_plot, labels=labels, patch_artist=True)



RESULTS - ALL LANGUAGES

Valid samples: 300/300

H1: SV → GEN-N (PSSR-PSSD)
  Mean: 77.7% (±4.0% SD)
  Median: 77.6%
  Range: [62.7%, 90.2%]
  Significant samples: 300/300 (100.0%)
  Result: ✓ SUPPORTED (mean > 50%)

H2: VS → N-GEN (PSSD-PSSR)
  Mean: 72.1% (±11.6% SD)
  Median: 72.7%
  Range: [40.0%, 100.0%]
  Significant samples: 58/297 (19.5%)
  Result: ✓ SUPPORTED (mean > 50%)

RESULTS - HEAD-MARKING LANGUAGES

Valid samples: 300/300

H1: SV → GEN-N (PSSR-PSSD)
  Mean: 81.7% (±7.7% SD)
  Significant samples: 279/300 (93.0%)

H2: VS → N-GEN (PSSD-PSSR)
  Mean: 68.4% (±17.2% SD)
  Significant samples: 22/300 (7.3%)

COMPARISON: ALL vs HEAD-MARKING

SV → GEN-N:
  All languages:  77.7%
  Head-marking:   81.7%
  Difference:     4.0 pp

VS → N-GEN:
  All languages:  72.1%
  Head-marking:   68.4%
  Difference:     3.8 pp

CREATING VISUALIZATIONS

Visualization saved to: sv_vs_gen_correlation_stratified.png
Detailed results (all) saved to: sv_vs_gen_all_languages.csv
Detailed results (hea

In [4]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import binomtest
from tqdm import tqdm


def cohens_d(prop1, prop2, n1, n2):
    """
    Calculate Cohen's d for two proportions.
    
    Cohen's d = (p1 - p2) / pooled_SD
    where pooled_SD = sqrt(p_pooled * (1 - p_pooled))
    """
    # Pooled proportion
    p_pooled = (prop1 * n1 + prop2 * n2) / (n1 + n2)
    
    # Pooled standard deviation
    pooled_sd = np.sqrt(p_pooled * (1 - p_pooled))
    
    # Cohen's d
    if pooled_sd == 0:
        return np.nan
    
    d = (prop1 - prop2) / pooled_sd
    return d


def analyze_sv_patterns_with_adjectives(data, sample_num=None, verbose=False):
    """
    Compare two models for SV languages:
    
    Model 1 (Simple): SV → GEN-N
    Model 2 (Complex): SV → (GEN-N OR (N-GEN AND N-Adj))
    
    Hypothesis: In SV languages with N-GEN, this results from N-raising,
    which should also produce N-Adj order.
    """
    
    data = data.copy()
    
    # GB130: SV/VS order
    data['SV_Order'] = 'Unknown'
    data.loc[data['GB130'] == '1', 'SV_Order'] = 'SV'
    data.loc[data['GB130'] == '2', 'SV_Order'] = 'VS'
    
    # GB065: Possessor order
    data['GEN_Order'] = 'Unknown'
    data.loc[data['GB065'] == '1', 'GEN_Order'] = 'GEN-N'
    data.loc[data['GB065'] == '2', 'GEN_Order'] = 'N-GEN'
    
    # GB193: Adjective order
    data['Adj_Order'] = 'Unknown'
    data.loc[data['GB193'] == '1', 'Adj_Order'] = 'Adj-N'
    data.loc[data['GB193'] == '2', 'Adj_Order'] = 'N-Adj'
    
    # Filter to SV languages with all features
    sv_langs = data[(data['SV_Order'] == 'SV') & 
                    (data['GEN_Order'] != 'Unknown') & 
                    (data['Adj_Order'] != 'Unknown')].copy()
    
    if len(sv_langs) < 10:
        return None
    
    # MODEL 1: Simple - SV → GEN-N
    model1_match = sv_langs['GEN_Order'] == 'GEN-N'
    model1_count = sum(model1_match)
    model1_prop = model1_count / len(sv_langs)
    
    # MODEL 2: Complex - SV → (GEN-N OR (N-GEN AND N-Adj))
    # Pattern is "expected" if:
    # - GEN-N (regardless of adjective order), OR
    # - N-GEN AND N-Adj (the N-raising pattern)
    model2_match = ((sv_langs['GEN_Order'] == 'GEN-N') | 
                    ((sv_langs['GEN_Order'] == 'N-GEN') & (sv_langs['Adj_Order'] == 'N-Adj')))
    model2_count = sum(model2_match)
    model2_prop = model2_count / len(sv_langs)
    
    # Calculate improvement
    improvement = model2_prop - model1_prop
    improvement_pct = improvement * 100
    
    # Cohen's d
    d = cohens_d(model2_prop, model1_prop, len(sv_langs), len(sv_langs))
    
    # Binomial tests (null: p = 0.5)
    binom1 = binomtest(model1_count, n=len(sv_langs), p=0.5, alternative='greater')
    binom2 = binomtest(model2_count, n=len(sv_langs), p=0.5, alternative='greater')
    
    # Break down the N-GEN cases
    n_gen_langs = sv_langs[sv_langs['GEN_Order'] == 'N-GEN']
    n_gen_total = len(n_gen_langs)
    n_gen_with_n_adj = sum((n_gen_langs['Adj_Order'] == 'N-Adj'))
    n_gen_with_adj_n = sum((n_gen_langs['Adj_Order'] == 'Adj-N'))
    
    if verbose:
        print(f"\nSample {sample_num}:")
        print(f"Total SV languages: {len(sv_langs)}")
        print(f"\nModel 1 (Simple): SV → GEN-N")
        print(f"  Matches: {model1_count}/{len(sv_langs)} ({model1_prop:.1%})")
        print(f"  p-value: {binom1.pvalue:.4f}")
        print(f"\nModel 2 (Complex): SV → (GEN-N OR (N-GEN AND N-Adj))")
        print(f"  Matches: {model2_count}/{len(sv_langs)} ({model2_prop:.1%})")
        print(f"  p-value: {binom2.pvalue:.4f}")
        print(f"\nImprovement: {improvement_pct:.1f} pp")
        print(f"Cohen's d: {d:.3f}")
        print(f"\nSV with N-GEN breakdown ({n_gen_total} languages):")
        print(f"  N-GEN + N-Adj (N-raising): {n_gen_with_n_adj} ({n_gen_with_n_adj/n_gen_total:.1%})")
        print(f"  N-GEN + Adj-N (unexpected): {n_gen_with_adj_n} ({n_gen_with_adj_n/n_gen_total:.1%})")
        print("-" * 70)
    
    return {
        'sample': sample_num,
        'n_sv': len(sv_langs),
        'model1_count': model1_count,
        'model1_prop': model1_prop,
        'model1_p': binom1.pvalue,
        'model2_count': model2_count,
        'model2_prop': model2_prop,
        'model2_p': binom2.pvalue,
        'improvement': improvement,
        'cohens_d': d,
        'n_gen_total': n_gen_total,
        'n_gen_n_adj': n_gen_with_n_adj,
        'n_gen_adj_n': n_gen_with_adj_n,
        'n_gen_n_adj_prop': n_gen_with_n_adj / n_gen_total if n_gen_total > 0 else np.nan
    }


def analyze_sv_patterns_head_marking(data, sample_num=None, verbose=False):
    """Same analysis for head-marking languages only"""
    
    if 'GB431' not in data.columns or 'GB433' not in data.columns:
        return None
    
    head_marking = data[(data['GB431'] == '1') | (data['GB433'] == '1')].copy()
    
    if len(head_marking) < 10:
        return None
    
    return analyze_sv_patterns_with_adjectives(head_marking, sample_num, verbose)


def create_stratified_sample(data, max_total=120, langs_per_area=20):
    """Create stratified sample following Hammarström & Donohue (2014)"""
    sampled_data = pd.DataFrame()
    
    macroareas = data['Macroarea'].dropna().unique()
    
    for area in macroareas:
        area_data = data[data['Macroarea'] == area].copy()
        if area_data.empty:
            continue
        
        families = area_data['Family'].unique()
        families = families[~pd.isna(families)]
        
        if len(families) == 0:
            continue
        
        num_to_sample = min(len(families), langs_per_area)
        selected_families = np.random.choice(families, size=num_to_sample, replace=False)
        
        for family in selected_families:
            family_data = area_data[area_data['Family'] == family]
            if not family_data.empty:
                sampled_lang = family_data.sample(n=1)
                sampled_data = pd.concat([sampled_data, sampled_lang])
    
    if len(sampled_data) > max_total:
        sampled_data = sampled_data.sample(n=max_total)
    
    return sampled_data


def visualize_results(all_langs_results, head_marking_results):
    """Create comprehensive visualizations"""
    
    all_df = pd.DataFrame([r for r in all_langs_results if r is not None])
    hm_df = pd.DataFrame([r for r in head_marking_results if r is not None])
    
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    
    # 1. Model comparison - All languages
    ax = axes[0, 0]
    if len(all_df) > 0:
        model1_props = all_df['model1_prop'].dropna() * 100
        model2_props = all_df['model2_prop'].dropna() * 100
        
        bp = ax.boxplot([model1_props, model2_props], 
                        labels=['Model 1:\nSV→GEN-N', 'Model 2:\nSV→(GEN-N OR\nN-GEN+N-Adj)'],
                        patch_artist=True)
        bp['boxes'][0].set_facecolor('lightblue')
        bp['boxes'][1].set_facecolor('lightcoral')
        
        ax.axhline(50, color='black', linestyle=':', linewidth=2, label='Chance')
        ax.set_ylabel('Proportion (%)', fontsize=11)
        ax.set_title('All Languages: Model Comparison', fontsize=12, fontweight='bold')
        ax.legend()
        ax.set_ylim(0, 100)
    
    # 2. Improvement distribution
    ax = axes[0, 1]
    if len(all_df) > 0:
        improvements = all_df['improvement'].dropna() * 100
        sns.histplot(improvements, kde=True, bins=20, ax=ax, color='green')
        ax.axvline(improvements.mean(), color='red', linestyle='--', linewidth=2,
                   label=f'Mean: {improvements.mean():.1f} pp')
        ax.axvline(0, color='black', linestyle=':', linewidth=2)
        ax.set_xlabel('Improvement (percentage points)', fontsize=11)
        ax.set_ylabel('Count', fontsize=11)
        ax.set_title('Model 2 Improvement over Model 1', fontsize=12, fontweight='bold')
        ax.legend()
    
    # 3. Cohen's d distribution
    ax = axes[0, 2]
    if len(all_df) > 0:
        cohens_ds = all_df['cohens_d'].dropna()
        sns.histplot(cohens_ds, kde=True, bins=20, ax=ax, color='purple')
        ax.axvline(cohens_ds.mean(), color='red', linestyle='--', linewidth=2,
                   label=f'Mean: {cohens_ds.mean():.3f}')
        ax.axvline(0.2, color='gray', linestyle=':', alpha=0.5, label='Small')
        ax.axvline(0.5, color='gray', linestyle='--', alpha=0.5, label='Medium')
        ax.axvline(0.8, color='gray', linestyle='-', alpha=0.5, label='Large')
        ax.set_xlabel("Cohen's d", fontsize=11)
        ax.set_ylabel('Count', fontsize=11)
        ax.set_title("Effect Size Distribution", fontsize=12, fontweight='bold')
        ax.legend(fontsize=8)
    
    # 4. N-GEN breakdown
    ax = axes[1, 0]
    if len(all_df) > 0:
        n_adj_props = all_df['n_gen_n_adj_prop'].dropna() * 100
        sns.histplot(n_adj_props, kde=True, bins=20, ax=ax, color='orange')
        ax.axvline(n_adj_props.mean(), color='red', linestyle='--', linewidth=2,
                   label=f'Mean: {n_adj_props.mean():.1f}%')
        ax.axvline(50, color='black', linestyle=':', linewidth=2, label='Chance')
        ax.set_xlabel('% of SV+N-GEN with N-Adj', fontsize=11)
        ax.set_ylabel('Count', fontsize=11)
        ax.set_title('N-raising Pattern in SV+N-GEN Languages', fontsize=12, fontweight='bold')
        ax.legend()
        ax.set_xlim(0, 100)
    
    # 5. Head-marking comparison
    ax = axes[1, 1]
    if len(all_df) > 0:
        data_all = [all_df['model1_prop'] * 100, all_df['model2_prop'] * 100]
        labels_all = ['Model 1\n(All)', 'Model 2\n(All)']
        colors = ['lightblue', 'lightcoral']
        
        if len(hm_df) > 0:
            data_all.extend([hm_df['model1_prop'] * 100, hm_df['model2_prop'] * 100])
            labels_all.extend(['Model 1\n(Head-Mrk)', 'Model 2\n(Head-Mrk)'])
            colors.extend(['skyblue', 'salmon'])
        
        bp = ax.boxplot(data_all, labels=labels_all, patch_artist=True)
        for patch, color in zip(bp['boxes'], colors):
            patch.set_facecolor(color)
        
        ax.axhline(50, color='black', linestyle=':', linewidth=2)
        ax.set_ylabel('Proportion (%)', fontsize=11)
        ax.set_title('All vs Head-Marking Comparison', fontsize=12, fontweight='bold')
        ax.set_ylim(0, 100)
    
    # 6. Effect size comparison
    ax = axes[1, 2]
    if len(all_df) > 0:
        mean_d_all = all_df['cohens_d'].mean()
        std_d_all = all_df['cohens_d'].std()
        
        x = [0]
        means = [mean_d_all]
        stds = [std_d_all]
        labels = ['All Languages']
        colors_bar = ['purple']
        
        if len(hm_df) > 0:
            mean_d_hm = hm_df['cohens_d'].mean()
            std_d_hm = hm_df['cohens_d'].std()
            x.append(1)
            means.append(mean_d_hm)
            stds.append(std_d_hm)
            labels.append('Head-Marking')
            colors_bar.append('orange')
        
        bars = ax.bar(x, means, yerr=stds, capsize=5, color=colors_bar, alpha=0.7)
        ax.set_xticks(x)
        ax.set_xticklabels(labels)
        ax.set_ylabel("Cohen's d", fontsize=11)
        ax.set_title('Effect Size: Adding Adjective Order', fontsize=12, fontweight='bold')
        ax.axhline(0.2, color='gray', linestyle=':', alpha=0.5, label='Small')
        ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='Medium')
        ax.axhline(0.8, color='gray', linestyle='-', alpha=0.5, label='Large')
        ax.legend(fontsize=8)
        ax.set_ylim(0, max(means) + max(stds) + 0.2)
    
    plt.tight_layout()
    plt.savefig('sv_adjective_effect_analysis.png', dpi=300, bbox_inches='tight')
    print("\nVisualization saved to: sv_adjective_effect_analysis.png")
    plt.close()


def main():
    # Set random seed
    np.random.seed(42)
    
    # Load data
    print("Loading data...")
    try:
        grambank = pd.read_csv('grambank_sane_format.csv')
        languages = pd.read_csv('languages1.csv')
    except FileNotFoundError as e:
        print(f"Error: {e}")
        return
    
    # Add metadata
    metadata = {}
    for _, row in languages.iterrows():
        if pd.notna(row['Name']):
            metadata[row['Name']] = {
                'macroarea': row['Macroarea'],
                'family': row['Family_name']
            }
    
    grambank['Macroarea'] = grambank['Language'].map(lambda x: metadata.get(x, {}).get('macroarea'))
    grambank['Family'] = grambank['Language'].map(lambda x: metadata.get(x, {}).get('family'))
    
    # Convert columns
    required_columns = ['GB130', 'GB065', 'GB193', 'GB431', 'GB433']
    for col in required_columns:
        if col in grambank.columns:
            grambank[col] = grambank[col].astype(str)
    
    print(f"Data loaded: {len(grambank)} languages")
    
    # Run analysis
    print("\n" + "="*70)
    print("TESTING N-RAISING HYPOTHESIS FOR SV LANGUAGES")
    print("="*70)
    print("\nTheoretical Hypothesis:")
    print("  SV languages with N-GEN (unexpected pattern) result from N-raising,")
    print("  which should also produce N-Adj order")
    print("\nModel Comparison:")
    print("  Model 1 (Simple):  SV → GEN-N")
    print("  Model 2 (Complex): SV → (GEN-N OR (N-GEN AND N-Adj))")
    print("\nEffect size: Cohen's d")
    
    n_samples = 300
    all_langs_results = []
    head_marking_results = []
    
    print("\nProcessing samples...")
    for i in tqdm(range(n_samples), desc="Stratified sampling"):
        sample = create_stratified_sample(grambank, max_total=120, langs_per_area=20)
        
        result_all = analyze_sv_patterns_with_adjectives(sample, sample_num=i+1, verbose=False)
        all_langs_results.append(result_all)
        
        result_hm = analyze_sv_patterns_head_marking(sample, sample_num=i+1, verbose=False)
        head_marking_results.append(result_hm)
    
    # Summary statistics
    print("\n" + "="*70)
    print("RESULTS - ALL LANGUAGES")
    print("="*70)
    
    valid_all = [r for r in all_langs_results if r is not None]
    
    if len(valid_all) > 0:
        model1_props = [r['model1_prop'] * 100 for r in valid_all]
        model2_props = [r['model2_prop'] * 100 for r in valid_all]
        improvements = [r['improvement'] * 100 for r in valid_all]
        cohens_ds = [r['cohens_d'] for r in valid_all if not np.isnan(r['cohens_d'])]
        n_adj_props = [r['n_gen_n_adj_prop'] * 100 for r in valid_all if not np.isnan(r['n_gen_n_adj_prop'])]
        
        print(f"\nValid samples: {len(valid_all)}/{n_samples}")
        
        print(f"\nModel 1 (Simple): SV → GEN-N")
        print(f"  Mean: {np.mean(model1_props):.1f}% (±{np.std(model1_props):.1f}%)")
        print(f"  Range: [{np.min(model1_props):.1f}%, {np.max(model1_props):.1f}%]")
        
        print(f"\nModel 2 (Complex): SV → (GEN-N OR (N-GEN AND N-Adj))")
        print(f"  Mean: {np.mean(model2_props):.1f}% (±{np.std(model2_props):.1f}%)")
        print(f"  Range: [{np.min(model2_props):.1f}%, {np.max(model2_props):.1f}%]")
        
        print(f"\nImprovement:")
        print(f"  Mean: {np.mean(improvements):.1f} percentage points (±{np.std(improvements):.1f})")
        print(f"  Range: [{np.min(improvements):.1f}, {np.max(improvements):.1f}] pp")
        
        print(f"\nEffect Size (Cohen's d):")
        print(f"  Mean: {np.mean(cohens_ds):.3f} (±{np.std(cohens_ds):.3f})")
        print(f"  Range: [{np.min(cohens_ds):.3f}, {np.max(cohens_ds):.3f}]")
        
        # Interpret Cohen's d
        mean_d = np.mean(cohens_ds)
        if mean_d < 0.2:
            interpretation = "negligible/small"
        elif mean_d < 0.5:
            interpretation = "small to medium"
        elif mean_d < 0.8:
            interpretation = "medium to large"
        else:
            interpretation = "large"
        print(f"  Interpretation: {interpretation} effect")
        
        print(f"\nN-raising Pattern (% of SV+N-GEN with N-Adj):")
        print(f"  Mean: {np.mean(n_adj_props):.1f}% (±{np.std(n_adj_props):.1f}%)")
        
        if np.mean(n_adj_props) > 50:
            print(f"  Result: ✓ N-raising hypothesis SUPPORTED")
        else:
            print(f"  Result: ✗ N-raising hypothesis NOT SUPPORTED")
    
    # Head-marking results
    print("\n" + "="*70)
    print("RESULTS - HEAD-MARKING LANGUAGES")
    print("="*70)
    
    valid_hm = [r for r in head_marking_results if r is not None]
    
    if len(valid_hm) > 0:
        model1_props_hm = [r['model1_prop'] * 100 for r in valid_hm]
        model2_props_hm = [r['model2_prop'] * 100 for r in valid_hm]
        improvements_hm = [r['improvement'] * 100 for r in valid_hm]
        cohens_ds_hm = [r['cohens_d'] for r in valid_hm if not np.isnan(r['cohens_d'])]
        
        print(f"\nValid samples: {len(valid_hm)}/{n_samples}")
        
        print(f"\nModel 1: {np.mean(model1_props_hm):.1f}% (±{np.std(model1_props_hm):.1f}%)")
        print(f"Model 2: {np.mean(model2_props_hm):.1f}% (±{np.std(model2_props_hm):.1f}%)")
        print(f"Improvement: {np.mean(improvements_hm):.1f} pp (±{np.std(improvements_hm):.1f})")
        print(f"Cohen's d: {np.mean(cohens_ds_hm):.3f} (±{np.std(cohens_ds_hm):.3f})")
        
        # Comparison
        print("\n" + "="*70)
        print("COMPARISON: ALL vs HEAD-MARKING")
        print("="*70)
        print(f"\nImprovement from adding adjective order:")
        print(f"  All languages:  {np.mean(improvements):.1f} pp")
        print(f"  Head-marking:   {np.mean(improvements_hm):.1f} pp")
        print(f"\nCohen's d:")
        print(f"  All languages:  {np.mean(cohens_ds):.3f}")
        print(f"  Head-marking:   {np.mean(cohens_ds_hm):.3f}")
    
    # Visualizations
    print("\n" + "="*70)
    print("CREATING VISUALIZATIONS")
    print("="*70)
    visualize_results(all_langs_results, head_marking_results)
    
    # Save results
    all_df = pd.DataFrame([r for r in all_langs_results if r is not None])
    all_df.to_csv('sv_adjective_effect_all.csv', index=False)
    print("Detailed results (all) saved to: sv_adjective_effect_all.csv")
    
    if len(valid_hm) > 0:
        hm_df = pd.DataFrame([r for r in head_marking_results if r is not None])
        hm_df.to_csv('sv_adjective_effect_head_marking.csv', index=False)
        print("Detailed results (head-marking) saved to: sv_adjective_effect_head_marking.csv")
    
    print("\n" + "="*70)
    print("ANALYSIS COMPLETE!")
    print("="*70)


if __name__ == "__main__":
    main()

Loading data...
Data loaded: 2467 languages

TESTING N-RAISING HYPOTHESIS FOR SV LANGUAGES

Theoretical Hypothesis:
  SV languages with N-GEN (unexpected pattern) result from N-raising,
  which should also produce N-Adj order

Model Comparison:
  Model 1 (Simple):  SV → GEN-N
  Model 2 (Complex): SV → (GEN-N OR (N-GEN AND N-Adj))

Effect size: Cohen's d

Processing samples...


Stratified sampling: 100%|████████████████████| 300/300 [00:09<00:00, 30.76it/s]
/var/folders/jk/cgmd03wn0g1b85mr8y_w4__c0000gn/T/ipykernel_11337/602398296.py:191: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot([model1_props, model2_props],
/var/folders/jk/cgmd03wn0g1b85mr8y_w4__c0000gn/T/ipykernel_11337/602398296.py:257: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(data_all, labels=labels_all, patch_artist=True)



RESULTS - ALL LANGUAGES

Valid samples: 300/300

Model 1 (Simple): SV → GEN-N
  Mean: 78.4% (±4.7%)
  Range: [56.1%, 93.0%]

Model 2 (Complex): SV → (GEN-N OR (N-GEN AND N-Adj))
  Mean: 97.6% (±2.1%)
  Range: [90.0%, 100.0%]

Improvement:
  Mean: 19.2 percentage points (±4.3)
  Range: [4.7, 36.6] pp

Effect Size (Cohen's d):
  Mean: 0.595 (±0.098)
  Range: [0.221, 0.860]
  Interpretation: medium to large effect

N-raising Pattern (% of SV+N-GEN with N-Adj):
  Mean: 89.3% (±9.2%)
  Result: ✓ N-raising hypothesis SUPPORTED

RESULTS - HEAD-MARKING LANGUAGES

Valid samples: 298/300

Model 1: 81.5% (±8.5%)
Model 2: 97.2% (±4.1%)
Improvement: 15.7 pp (±7.9)
Cohen's d: 0.520 (±0.186)

COMPARISON: ALL vs HEAD-MARKING

Improvement from adding adjective order:
  All languages:  19.2 pp
  Head-marking:   15.7 pp

Cohen's d:
  All languages:  0.595
  Head-marking:   0.520

CREATING VISUALIZATIONS

Visualization saved to: sv_adjective_effect_analysis.png
Detailed results (all) saved to: sv_adjecti

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import binomtest
from scipy import stats
from tqdm import tqdm


def cohens_d(prop1, prop2, n1, n2):
    """
    Calculate Cohen's d for two proportions.
    
    Cohen's d = (p1 - p2) / pooled_SD
    where pooled_SD = sqrt(p_pooled * (1 - p_pooled))
    """
    # Pooled proportion
    p_pooled = (prop1 * n1 + prop2 * n2) / (n1 + n2)
    
    # Pooled standard deviation
    pooled_sd = np.sqrt(p_pooled * (1 - p_pooled))
    
    # Cohen's d
    if pooled_sd == 0:
        return np.nan
    
    d = (prop1 - prop2) / pooled_sd
    return d


def analyze_sv_patterns_with_adjectives(data, sample_num=None, verbose=False):
    """
    Compare two models for SV languages:
    
    Model 1 (Simple): SV → GEN-N
    Model 2 (Complex): SV → (GEN-N OR (N-GEN AND N-Adj))
    
    Hypothesis: In SV languages with N-GEN, this results from N-raising,
    which should also produce N-Adj order.
    """
    
    data = data.copy()
    
    # GB130: SV/VS order
    data['SV_Order'] = 'Unknown'
    data.loc[data['GB130'] == '1', 'SV_Order'] = 'SV'
    data.loc[data['GB130'] == '2', 'SV_Order'] = 'VS'
    
    # GB065: Possessor order
    data['GEN_Order'] = 'Unknown'
    data.loc[data['GB065'] == '1', 'GEN_Order'] = 'GEN-N'
    data.loc[data['GB065'] == '2', 'GEN_Order'] = 'N-GEN'
    
    # GB193: Adjective order
    data['Adj_Order'] = 'Unknown'
    data.loc[data['GB193'] == '1', 'Adj_Order'] = 'Adj-N'
    data.loc[data['GB193'] == '2', 'Adj_Order'] = 'N-Adj'
    
    # Filter to SV languages with all features
    sv_langs = data[(data['SV_Order'] == 'SV') & 
                    (data['GEN_Order'] != 'Unknown') & 
                    (data['Adj_Order'] != 'Unknown')].copy()
    
    if len(sv_langs) < 10:
        return None
    
    # MODEL 1: Simple - SV → GEN-N
    model1_match = sv_langs['GEN_Order'] == 'GEN-N'
    model1_count = sum(model1_match)
    model1_prop = model1_count / len(sv_langs)
    
    # MODEL 2: Complex - SV → (GEN-N OR (N-GEN AND N-Adj))
    # Pattern is "expected" if:
    # - GEN-N (regardless of adjective order), OR
    # - N-GEN AND N-Adj (the N-raising pattern)
    model2_match = ((sv_langs['GEN_Order'] == 'GEN-N') | 
                    ((sv_langs['GEN_Order'] == 'N-GEN') & (sv_langs['Adj_Order'] == 'N-Adj')))
    model2_count = sum(model2_match)
    model2_prop = model2_count / len(sv_langs)
    
    # Calculate improvement
    improvement = model2_prop - model1_prop
    improvement_pct = improvement * 100
    
    # Cohen's d
    d = cohens_d(model2_prop, model1_prop, len(sv_langs), len(sv_langs))
    
    # Binomial tests (null: p = 0.5)
    binom1 = binomtest(model1_count, n=len(sv_langs), p=0.5, alternative='greater')
    binom2 = binomtest(model2_count, n=len(sv_langs), p=0.5, alternative='greater')
    
    # Break down the N-GEN cases
    n_gen_langs = sv_langs[sv_langs['GEN_Order'] == 'N-GEN']
    n_gen_total = len(n_gen_langs)
    n_gen_with_n_adj = sum((n_gen_langs['Adj_Order'] == 'N-Adj'))
    n_gen_with_adj_n = sum((n_gen_langs['Adj_Order'] == 'Adj-N'))
    
    if verbose:
        print(f"\nSample {sample_num}:")
        print(f"Total SV languages: {len(sv_langs)}")
        print(f"\nModel 1 (Simple): SV → GEN-N")
        print(f"  Matches: {model1_count}/{len(sv_langs)} ({model1_prop:.1%})")
        print(f"  p-value: {binom1.pvalue:.4f}")
        print(f"\nModel 2 (Complex): SV → (GEN-N OR (N-GEN AND N-Adj))")
        print(f"  Matches: {model2_count}/{len(sv_langs)} ({model2_prop:.1%})")
        print(f"  p-value: {binom2.pvalue:.4f}")
        print(f"\nImprovement: {improvement_pct:.1f} pp")
        print(f"Cohen's d: {d:.3f}")
        print(f"\nSV with N-GEN breakdown ({n_gen_total} languages):")
        print(f"  N-GEN + N-Adj (N-raising): {n_gen_with_n_adj} ({n_gen_with_n_adj/n_gen_total:.1%})")
        print(f"  N-GEN + Adj-N (unexpected): {n_gen_with_adj_n} ({n_gen_with_adj_n/n_gen_total:.1%})")
        print("-" * 70)
    
    return {
        'sample': sample_num,
        'n_sv': len(sv_langs),
        'model1_count': model1_count,
        'model1_prop': model1_prop,
        'model1_p': binom1.pvalue,
        'model2_count': model2_count,
        'model2_prop': model2_prop,
        'model2_p': binom2.pvalue,
        'improvement': improvement,
        'cohens_d': d,
        'n_gen_total': n_gen_total,
        'n_gen_n_adj': n_gen_with_n_adj,
        'n_gen_adj_n': n_gen_with_adj_n,
        'n_gen_n_adj_prop': n_gen_with_n_adj / n_gen_total if n_gen_total > 0 else np.nan
    }


def analyze_sv_patterns_head_marking(data, sample_num=None, verbose=False):
    """Same analysis for head-marking languages only"""
    
    if 'GB431' not in data.columns or 'GB433' not in data.columns:
        return None
    
    head_marking = data[(data['GB431'] == '1') | (data['GB433'] == '1')].copy()
    
    if len(head_marking) < 10:
        return None
    
    return analyze_sv_patterns_with_adjectives(head_marking, sample_num, verbose)


def create_stratified_sample(data, max_total=120, langs_per_area=20):
    """Create stratified sample following Hammarström & Donohue (2014)"""
    sampled_data = pd.DataFrame()
    
    macroareas = data['Macroarea'].dropna().unique()
    
    for area in macroareas:
        area_data = data[data['Macroarea'] == area].copy()
        if area_data.empty:
            continue
        
        families = area_data['Family'].unique()
        families = families[~pd.isna(families)]
        
        if len(families) == 0:
            continue
        
        num_to_sample = min(len(families), langs_per_area)
        selected_families = np.random.choice(families, size=num_to_sample, replace=False)
        
        for family in selected_families:
            family_data = area_data[area_data['Family'] == family]
            if not family_data.empty:
                sampled_lang = family_data.sample(n=1)
                sampled_data = pd.concat([sampled_data, sampled_lang])
    
    if len(sampled_data) > max_total:
        sampled_data = sampled_data.sample(n=max_total)
    
    return sampled_data


def calculate_ci(data, confidence=0.95):
    """Calculate confidence interval for data"""
    n = len(data)
    mean = np.mean(data)
    se = np.std(data, ddof=1) / np.sqrt(n)
    
    # Use t-distribution for small samples
    from scipy import stats
    t_crit = stats.t.ppf((1 + confidence) / 2, n - 1)
    ci_lower = mean - t_crit * se
    ci_upper = mean + t_crit * se
    
    return mean, ci_lower, ci_upper, se


def visualize_results(all_langs_results, head_marking_results):
    """Create comprehensive visualizations with confidence intervals"""
    
    all_df = pd.DataFrame([r for r in all_langs_results if r is not None])
    hm_df = pd.DataFrame([r for r in head_marking_results if r is not None])
    
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    
    # 1. Model comparison with confidence intervals - All languages
    ax = axes[0, 0]
    if len(all_df) > 0:
        model1_props = all_df['model1_prop'].dropna() * 100
        model2_props = all_df['model2_prop'].dropna() * 100
        
        # Calculate means and CIs
        m1_mean, m1_lower, m1_upper, m1_se = calculate_ci(model1_props)
        m2_mean, m2_lower, m2_upper, m2_se = calculate_ci(model2_props)
        
        x = [0, 1]
        means = [m1_mean, m2_mean]
        ci_lower = [m1_lower, m2_lower]
        ci_upper = [m1_upper, m2_upper]
        colors = ['lightblue', 'lightcoral']
        
        # Bar plot with CI error bars
        bars = ax.bar(x, means, color=colors, alpha=0.7, width=0.6)
        ax.errorbar(x, means, 
                   yerr=[[m1_mean - m1_lower, m2_mean - m2_lower],
                         [m1_upper - m1_mean, m2_upper - m2_mean]],
                   fmt='none', color='black', capsize=8, linewidth=2, label='95% CI')
        
        # Add mean values on bars
        for i, (bar, mean) in enumerate(zip(bars, means)):
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height + 2,
                   f'{mean:.1f}%\n[{ci_lower[i]:.1f}, {ci_upper[i]:.1f}]',
                   ha='center', va='bottom', fontsize=9, fontweight='bold')
        
        ax.axhline(50, color='red', linestyle=':', linewidth=2, alpha=0.5, label='Chance')
        ax.set_ylabel('Proportion (%)', fontsize=11)
        ax.set_title('All Languages: Model Comparison\n(with 95% CI)', fontsize=12, fontweight='bold')
        ax.set_xticks(x)
        ax.set_xticklabels(['Model 1:\nSV→GEN-N', 'Model 2:\nSV→(GEN-N OR\nN-GEN+N-Adj)'])
        ax.legend(loc='lower right', fontsize=8)
        ax.set_ylim(0, 105)
    
    # 2. Improvement distribution with CI
    ax = axes[0, 1]
    if len(all_df) > 0:
        improvements = all_df['improvement'].dropna() * 100
        imp_mean, imp_lower, imp_upper, imp_se = calculate_ci(improvements)
        
        sns.histplot(improvements, kde=True, bins=20, ax=ax, color='green', alpha=0.6)
        ax.axvline(imp_mean, color='red', linestyle='--', linewidth=2,
                   label=f'Mean: {imp_mean:.1f} pp')
        ax.axvline(imp_lower, color='red', linestyle=':', linewidth=1.5, alpha=0.7,
                   label=f'95% CI: [{imp_lower:.1f}, {imp_upper:.1f}]')
        ax.axvline(imp_upper, color='red', linestyle=':', linewidth=1.5, alpha=0.7)
        ax.axvline(0, color='black', linestyle=':', linewidth=2)
        ax.set_xlabel('Improvement (percentage points)', fontsize=11)
        ax.set_ylabel('Count', fontsize=11)
        ax.set_title('Model 2 Improvement over Model 1\n(with 95% CI)', fontsize=12, fontweight='bold')
        ax.legend(fontsize=8)
    
    # 3. Cohen's d distribution with CI
    ax = axes[0, 2]
    if len(all_df) > 0:
        cohens_ds = all_df['cohens_d'].dropna()
        d_mean, d_lower, d_upper, d_se = calculate_ci(cohens_ds)
        
        sns.histplot(cohens_ds, kde=True, bins=20, ax=ax, color='purple', alpha=0.6)
        ax.axvline(d_mean, color='red', linestyle='--', linewidth=2,
                   label=f'Mean: {d_mean:.3f}')
        ax.axvline(d_lower, color='red', linestyle=':', linewidth=1.5, alpha=0.7,
                   label=f'95% CI: [{d_lower:.3f}, {d_upper:.3f}]')
        ax.axvline(d_upper, color='red', linestyle=':', linewidth=1.5, alpha=0.7)
        ax.axvline(0.2, color='gray', linestyle=':', alpha=0.4, linewidth=1)
        ax.axvline(0.5, color='gray', linestyle='--', alpha=0.4, linewidth=1)
        ax.axvline(0.8, color='gray', linestyle='-', alpha=0.4, linewidth=1)
        ax.text(0.2, ax.get_ylim()[1]*0.9, 'Small', fontsize=7, alpha=0.6)
        ax.text(0.5, ax.get_ylim()[1]*0.9, 'Medium', fontsize=7, alpha=0.6)
        ax.text(0.8, ax.get_ylim()[1]*0.9, 'Large', fontsize=7, alpha=0.6)
        ax.set_xlabel("Cohen's d", fontsize=11)
        ax.set_ylabel('Count', fontsize=11)
        ax.set_title("Effect Size Distribution\n(with 95% CI)", fontsize=12, fontweight='bold')
        ax.legend(fontsize=8)
    
    # 4. N-GEN breakdown with CI
    ax = axes[1, 0]
    if len(all_df) > 0:
        n_adj_props = all_df['n_gen_n_adj_prop'].dropna() * 100
        n_mean, n_lower, n_upper, n_se = calculate_ci(n_adj_props)
        
        sns.histplot(n_adj_props, kde=True, bins=20, ax=ax, color='orange', alpha=0.6)
        ax.axvline(n_mean, color='red', linestyle='--', linewidth=2,
                   label=f'Mean: {n_mean:.1f}%')
        ax.axvline(n_lower, color='red', linestyle=':', linewidth=1.5, alpha=0.7,
                   label=f'95% CI: [{n_lower:.1f}, {n_upper:.1f}]')
        ax.axvline(n_upper, color='red', linestyle=':', linewidth=1.5, alpha=0.7)
        ax.axvline(50, color='black', linestyle=':', linewidth=2, label='Chance')
        ax.set_xlabel('% of SV+N-GEN with N-Adj', fontsize=11)
        ax.set_ylabel('Count', fontsize=11)
        ax.set_title('N-raising Pattern in SV+N-GEN\n(with 95% CI)', fontsize=12, fontweight='bold')
        ax.legend(fontsize=8)
        ax.set_xlim(0, 100)
    
    # 5. Head-marking comparison with CI
    ax = axes[1, 1]
    if len(all_df) > 0:
        # Calculate CIs for all languages
        m1_all_mean, m1_all_lower, m1_all_upper, _ = calculate_ci(all_df['model1_prop'] * 100)
        m2_all_mean, m2_all_lower, m2_all_upper, _ = calculate_ci(all_df['model2_prop'] * 100)
        
        x_pos = [0, 1]
        means = [m1_all_mean, m2_all_mean]
        ci_errors = [[m1_all_mean - m1_all_lower, m2_all_mean - m2_all_lower],
                     [m1_all_upper - m1_all_mean, m2_all_upper - m2_all_mean]]
        colors = ['lightblue', 'lightcoral']
        labels = ['Model 1\n(All)', 'Model 2\n(All)']
        
        if len(hm_df) > 0:
            m1_hm_mean, m1_hm_lower, m1_hm_upper, _ = calculate_ci(hm_df['model1_prop'] * 100)
            m2_hm_mean, m2_hm_lower, m2_hm_upper, _ = calculate_ci(hm_df['model2_prop'] * 100)
            
            x_pos.extend([3, 4])
            means.extend([m1_hm_mean, m2_hm_mean])
            ci_errors[0].extend([m1_hm_mean - m1_hm_lower, m2_hm_mean - m2_hm_lower])
            ci_errors[1].extend([m1_hm_upper - m1_hm_mean, m2_hm_upper - m2_hm_mean])
            colors.extend(['skyblue', 'salmon'])
            labels.extend(['Model 1\n(Head-Mrk)', 'Model 2\n(Head-Mrk)'])
        
        bars = ax.bar(x_pos, means, color=colors, alpha=0.7, width=0.7)
        ax.errorbar(x_pos, means, yerr=ci_errors, fmt='none', color='black', 
                   capsize=6, linewidth=1.5, label='95% CI')
        
        # Add values on bars
        for i, (bar, mean) in enumerate(zip(bars, means)):
            ax.text(bar.get_x() + bar.get_width()/2., mean + 2,
                   f'{mean:.1f}%', ha='center', va='bottom', fontsize=8, fontweight='bold')
        
        ax.axhline(50, color='red', linestyle=':', linewidth=2, alpha=0.5)
        ax.set_ylabel('Proportion (%)', fontsize=11)
        ax.set_title('All vs Head-Marking Comparison\n(with 95% CI)', fontsize=12, fontweight='bold')
        ax.set_xticks(x_pos)
        ax.set_xticklabels(labels, fontsize=9)
        ax.legend(fontsize=8)
        ax.set_ylim(0, 100)
    
    # 6. Effect size comparison with CI
    ax = axes[1, 2]
    if len(all_df) > 0:
        d_all_mean, d_all_lower, d_all_upper, _ = calculate_ci(all_df['cohens_d'].dropna())
        
        x = [0]
        means = [d_all_mean]
        ci_errors = [[d_all_mean - d_all_lower], [d_all_upper - d_all_mean]]
        labels = ['All Languages']
        colors_bar = ['purple']
        
        if len(hm_df) > 0:
            d_hm_mean, d_hm_lower, d_hm_upper, _ = calculate_ci(hm_df['cohens_d'].dropna())
            x.append(1)
            means.append(d_hm_mean)
            ci_errors[0].append(d_hm_mean - d_hm_lower)
            ci_errors[1].append(d_hm_upper - d_hm_mean)
            labels.append('Head-Marking')
            colors_bar.append('orange')
        
        bars = ax.bar(x, means, color=colors_bar, alpha=0.7, width=0.5)
        ax.errorbar(x, means, yerr=ci_errors, fmt='none', color='black', 
                   capsize=8, linewidth=2, label='95% CI')
        
        # Add values on bars
        for i, (bar, mean, lower, upper) in enumerate(zip(bars, means, 
                                                          [d_all_lower] + ([d_hm_lower] if len(hm_df) > 0 else []),
                                                          [d_all_upper] + ([d_hm_upper] if len(hm_df) > 0 else []))):
            ax.text(bar.get_x() + bar.get_width()/2., mean + 0.05,
                   f'{mean:.3f}\n[{lower:.3f},\n{upper:.3f}]',
                   ha='center', va='bottom', fontsize=8, fontweight='bold')
        
        ax.set_xticks(x)
        ax.set_xticklabels(labels)
        ax.set_ylabel("Cohen's d", fontsize=11)
        ax.set_title('Effect Size: Adding Adjective Order\n(with 95% CI)', fontsize=12, fontweight='bold')
        ax.axhline(0.2, color='gray', linestyle=':', alpha=0.4, linewidth=1)
        ax.axhline(0.5, color='gray', linestyle='--', alpha=0.4, linewidth=1)
        ax.axhline(0.8, color='gray', linestyle='-', alpha=0.4, linewidth=1)
        ax.text(max(x) * 0.7, 0.2, 'Small', fontsize=7, alpha=0.6)
        ax.text(max(x) * 0.7, 0.5, 'Medium', fontsize=7, alpha=0.6)
        ax.text(max(x) * 0.7, 0.8, 'Large', fontsize=7, alpha=0.6)
        ax.legend(fontsize=8)
        ax.set_ylim(0, max(means) + max([ci_errors[1][i] for i in range(len(means))]) + 0.3)
    
    plt.tight_layout()
    plt.savefig('sv_adjective_effect_analysis.png', dpi=300, bbox_inches='tight')
    print("\nVisualization saved to: sv_adjective_effect_analysis.png")
    plt.close()


def main():
    # Set random seed
    np.random.seed(42)
    
    # Load data
    print("Loading data...")
    try:
        grambank = pd.read_csv('grambank_sane_format.csv')
        languages = pd.read_csv('languages1.csv')
    except FileNotFoundError as e:
        print(f"Error: {e}")
        return
    
    # Add metadata
    metadata = {}
    for _, row in languages.iterrows():
        if pd.notna(row['Name']):
            metadata[row['Name']] = {
                'macroarea': row['Macroarea'],
                'family': row['Family_name']
            }
    
    grambank['Macroarea'] = grambank['Language'].map(lambda x: metadata.get(x, {}).get('macroarea'))
    grambank['Family'] = grambank['Language'].map(lambda x: metadata.get(x, {}).get('family'))
    
    # Convert columns
    required_columns = ['GB130', 'GB065', 'GB193', 'GB431', 'GB433']
    for col in required_columns:
        if col in grambank.columns:
            grambank[col] = grambank[col].astype(str)
    
    print(f"Data loaded: {len(grambank)} languages")
    
    # Run analysis
    print("\n" + "="*70)
    print("TESTING N-RAISING HYPOTHESIS FOR SV LANGUAGES")
    print("="*70)
    print("\nTheoretical Hypothesis:")
    print("  SV languages with N-GEN (unexpected pattern) result from N-raising,")
    print("  which should also produce N-Adj order")
    print("\nModel Comparison:")
    print("  Model 1 (Simple):  SV → GEN-N")
    print("  Model 2 (Complex): SV → (GEN-N OR (N-GEN AND N-Adj))")
    print("\nEffect size: Cohen's d")
    
    n_samples = 300
    all_langs_results = []
    head_marking_results = []
    
    print("\nProcessing samples...")
    for i in tqdm(range(n_samples), desc="Stratified sampling"):
        sample = create_stratified_sample(grambank, max_total=120, langs_per_area=20)
        
        result_all = analyze_sv_patterns_with_adjectives(sample, sample_num=i+1, verbose=False)
        all_langs_results.append(result_all)
        
        result_hm = analyze_sv_patterns_head_marking(sample, sample_num=i+1, verbose=False)
        head_marking_results.append(result_hm)
    
    # Summary statistics
    print("\n" + "="*70)
    print("RESULTS - ALL LANGUAGES")
    print("="*70)
    
    valid_all = [r for r in all_langs_results if r is not None]
    
    if len(valid_all) > 0:
        model1_props = [r['model1_prop'] * 100 for r in valid_all]
        model2_props = [r['model2_prop'] * 100 for r in valid_all]
        improvements = [r['improvement'] * 100 for r in valid_all]
        cohens_ds = [r['cohens_d'] for r in valid_all if not np.isnan(r['cohens_d'])]
        n_adj_props = [r['n_gen_n_adj_prop'] * 100 for r in valid_all if not np.isnan(r['n_gen_n_adj_prop'])]
        
        # Calculate confidence intervals
        m1_mean, m1_lower, m1_upper, _ = calculate_ci(model1_props)
        m2_mean, m2_lower, m2_upper, _ = calculate_ci(model2_props)
        imp_mean, imp_lower, imp_upper, _ = calculate_ci(improvements)
        d_mean, d_lower, d_upper, _ = calculate_ci(cohens_ds)
        n_mean, n_lower, n_upper, _ = calculate_ci(n_adj_props)
        
        print(f"\nValid samples: {len(valid_all)}/{n_samples}")
        
        print(f"\nModel 1 (Simple): SV → GEN-N")
        print(f"  Mean: {m1_mean:.1f}% (±{np.std(model1_props):.1f}%)")
        print(f"  95% CI: [{m1_lower:.1f}%, {m1_upper:.1f}%]")
        print(f"  Range: [{np.min(model1_props):.1f}%, {np.max(model1_props):.1f}%]")
        
        print(f"\nModel 2 (Complex): SV → (GEN-N OR (N-GEN AND N-Adj))")
        print(f"  Mean: {m2_mean:.1f}% (±{np.std(model2_props):.1f}%)")
        print(f"  95% CI: [{m2_lower:.1f}%, {m2_upper:.1f}%]")
        print(f"  Range: [{np.min(model2_props):.1f}%, {np.max(model2_props):.1f}%]")
        
        print(f"\nImprovement:")
        print(f"  Mean: {imp_mean:.1f} percentage points (±{np.std(improvements):.1f})")
        print(f"  95% CI: [{imp_lower:.1f}, {imp_upper:.1f}] pp")
        print(f"  Range: [{np.min(improvements):.1f}, {np.max(improvements):.1f}] pp")
        
        print(f"\nEffect Size (Cohen's d):")
        print(f"  Mean: {d_mean:.3f} (±{np.std(cohens_ds):.3f})")
        print(f"  95% CI: [{d_lower:.3f}, {d_upper:.3f}]")
        print(f"  Range: [{np.min(cohens_ds):.3f}, {np.max(cohens_ds):.3f}]")
        
        # Interpret Cohen's d with CI
        if d_lower < 0.2:
            interpretation = "negligible/small"
        elif d_lower < 0.5:
            interpretation = "small to medium"
        elif d_lower < 0.8:
            interpretation = "medium to large"
        else:
            interpretation = "large"
        print(f"  Interpretation: {interpretation} effect (95% CI lower bound)")
        
        print(f"\nN-raising Pattern (% of SV+N-GEN with N-Adj):")
        print(f"  Mean: {n_mean:.1f}% (±{np.std(n_adj_props):.1f}%)")
        print(f"  95% CI: [{n_lower:.1f}%, {n_upper:.1f}%]")
        
        if n_lower > 50:
            print(f"  Result: ✓ N-raising hypothesis STRONGLY SUPPORTED (CI lower > 50%)")
        elif n_mean > 50:
            print(f"  Result: ⚠ N-raising hypothesis WEAKLY SUPPORTED (mean > 50%, but CI includes 50%)")
        else:
            print(f"  Result: ✗ N-raising hypothesis NOT SUPPORTED")
    
    # Head-marking results
    print("\n" + "="*70)
    print("RESULTS - HEAD-MARKING LANGUAGES")
    print("="*70)
    
    valid_hm = [r for r in head_marking_results if r is not None]
    
    if len(valid_hm) > 0:
        model1_props_hm = [r['model1_prop'] * 100 for r in valid_hm]
        model2_props_hm = [r['model2_prop'] * 100 for r in valid_hm]
        improvements_hm = [r['improvement'] * 100 for r in valid_hm]
        cohens_ds_hm = [r['cohens_d'] for r in valid_hm if not np.isnan(r['cohens_d'])]
        
        # Calculate CIs
        m1_hm_mean, m1_hm_lower, m1_hm_upper, _ = calculate_ci(model1_props_hm)
        m2_hm_mean, m2_hm_lower, m2_hm_upper, _ = calculate_ci(model2_props_hm)
        imp_hm_mean, imp_hm_lower, imp_hm_upper, _ = calculate_ci(improvements_hm)
        d_hm_mean, d_hm_lower, d_hm_upper, _ = calculate_ci(cohens_ds_hm)
        
        print(f"\nValid samples: {len(valid_hm)}/{n_samples}")
        
        print(f"\nModel 1: {m1_hm_mean:.1f}% (±{np.std(model1_props_hm):.1f}%)")
        print(f"  95% CI: [{m1_hm_lower:.1f}%, {m1_hm_upper:.1f}%]")
        
        print(f"\nModel 2: {m2_hm_mean:.1f}% (±{np.std(model2_props_hm):.1f}%)")
        print(f"  95% CI: [{m2_hm_lower:.1f}%, {m2_hm_upper:.1f}%]")
        
        print(f"\nImprovement: {imp_hm_mean:.1f} pp (±{np.std(improvements_hm):.1f})")
        print(f"  95% CI: [{imp_hm_lower:.1f}, {imp_hm_upper:.1f}] pp")
        
        print(f"\nCohen's d: {d_hm_mean:.3f} (±{np.std(cohens_ds_hm):.3f})")
        print(f"  95% CI: [{d_hm_lower:.3f}, {d_hm_upper:.3f}]")
        
        # Comparison
        print("\n" + "="*70)
        print("COMPARISON: ALL vs HEAD-MARKING")
        print("="*70)
        print(f"\nModel 1 accuracy:")
        print(f"  All languages:  {m1_mean:.1f}% [{m1_lower:.1f}, {m1_upper:.1f}]")
        print(f"  Head-marking:   {m1_hm_mean:.1f}% [{m1_hm_lower:.1f}, {m1_hm_upper:.1f}]")
        
        print(f"\nModel 2 accuracy:")
        print(f"  All languages:  {m2_mean:.1f}% [{m2_lower:.1f}, {m2_upper:.1f}]")
        print(f"  Head-marking:   {m2_hm_mean:.1f}% [{m2_hm_lower:.1f}, {m2_hm_upper:.1f}]")
        
        print(f"\nImprovement from adding adjective order:")
        print(f"  All languages:  {imp_mean:.1f} pp [{imp_lower:.1f}, {imp_upper:.1f}]")
        print(f"  Head-marking:   {imp_hm_mean:.1f} pp [{imp_hm_lower:.1f}, {imp_hm_upper:.1f}]")
        
        print(f"\nCohen's d:")
        print(f"  All languages:  {d_mean:.3f} [{d_lower:.3f}, {d_upper:.3f}]")
        print(f"  Head-marking:   {d_hm_mean:.3f} [{d_hm_lower:.3f}, {d_hm_upper:.3f}]")
    
    # Visualizations
    print("\n" + "="*70)
    print("CREATING VISUALIZATIONS")
    print("="*70)
    visualize_results(all_langs_results, head_marking_results)
    
    # Save results
    all_df = pd.DataFrame([r for r in all_langs_results if r is not None])
    all_df.to_csv('sv_adjective_effect_all.csv', index=False)
    print("Detailed results (all) saved to: sv_adjective_effect_all.csv")
    
    if len(valid_hm) > 0:
        hm_df = pd.DataFrame([r for r in head_marking_results if r is not None])
        hm_df.to_csv('sv_adjective_effect_head_marking.csv', index=False)
        print("Detailed results (head-marking) saved to: sv_adjective_effect_head_marking.csv")
    
    print("\n" + "="*70)
    print("ANALYSIS COMPLETE!")
    print("="*70)


if __name__ == "__main__":
    main()

Loading data...
Data loaded: 2467 languages

TESTING N-RAISING HYPOTHESIS FOR SV LANGUAGES

Theoretical Hypothesis:
  SV languages with N-GEN (unexpected pattern) result from N-raising,
  which should also produce N-Adj order

Model Comparison:
  Model 1 (Simple):  SV → GEN-N
  Model 2 (Complex): SV → (GEN-N OR (N-GEN AND N-Adj))

Effect size: Cohen's d

Processing samples...


Stratified sampling: 100%|████████████████████| 300/300 [00:10<00:00, 29.56it/s]



RESULTS - ALL LANGUAGES

Valid samples: 300/300

Model 1 (Simple): SV → GEN-N
  Mean: 78.4% (±4.7%)
  95% CI: [77.8%, 78.9%]
  Range: [56.1%, 93.0%]

Model 2 (Complex): SV → (GEN-N OR (N-GEN AND N-Adj))
  Mean: 97.6% (±2.1%)
  95% CI: [97.4%, 97.9%]
  Range: [90.0%, 100.0%]

Improvement:
  Mean: 19.2 percentage points (±4.3)
  95% CI: [18.8, 19.7] pp
  Range: [4.7, 36.6] pp

Effect Size (Cohen's d):
  Mean: 0.595 (±0.098)
  95% CI: [0.583, 0.606]
  Range: [0.221, 0.860]
  Interpretation: medium to large effect (95% CI lower bound)

N-raising Pattern (% of SV+N-GEN with N-Adj):
  Mean: 89.3% (±9.2%)
  95% CI: [88.3%, 90.3%]
  Result: ✓ N-raising hypothesis STRONGLY SUPPORTED (CI lower > 50%)

RESULTS - HEAD-MARKING LANGUAGES

Valid samples: 298/300

Model 1: 81.5% (±8.5%)
  95% CI: [80.5%, 82.5%]

Model 2: 97.2% (±4.1%)
  95% CI: [96.7%, 97.6%]

Improvement: 15.7 pp (±7.9)
  95% CI: [14.8, 16.6] pp

Cohen's d: 0.520 (±0.186)
  95% CI: [0.498, 0.541]

COMPARISON: ALL vs HEAD-MARKING

Mo

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import binomtest, chi2_contingency
from scipy import stats


def wilson_ci(successes, n, confidence=0.95):
    """
    Calculate Wilson score confidence interval for a proportion.
    More accurate than normal approximation, especially for extreme proportions.
    """
    if n == 0:
        return 0, 0, 0
    
    p = successes / n
    z = stats.norm.ppf((1 + confidence) / 2)
    
    denominator = 1 + z**2 / n
    center = (p + z**2 / (2*n)) / denominator
    margin = z * np.sqrt(p*(1-p)/n + z**2/(4*n**2)) / denominator
    
    ci_lower = max(0, center - margin)
    ci_upper = min(1, center + margin)
    
    return p, ci_lower, ci_upper


def cohens_d(prop1, prop2, n1, n2):
    """Calculate Cohen's d for two proportions"""
    p_pooled = (prop1 * n1 + prop2 * n2) / (n1 + n2)
    pooled_sd = np.sqrt(p_pooled * (1 - p_pooled))
    
    if pooled_sd == 0:
        return np.nan
    
    return (prop1 - prop2) / pooled_sd


def analyze_full_dataset(data):
    """Analyze the full dataset without sampling"""
    
    print("\n" + "="*70)
    print("ANALYZING FULL DATASET")
    print("="*70)
    
    # GB130: SV/VS order
    data['SV_Order'] = 'Unknown'
    data.loc[data['GB130'] == '1', 'SV_Order'] = 'SV'
    data.loc[data['GB130'] == '2', 'SV_Order'] = 'VS'
    
    # GB065: Possessor order
    data['GEN_Order'] = 'Unknown'
    data.loc[data['GB065'] == '1', 'GEN_Order'] = 'GEN-N'
    data.loc[data['GB065'] == '2', 'GEN_Order'] = 'N-GEN'
    
    # GB193: Adjective order
    data['Adj_Order'] = 'Unknown'
    data.loc[data['GB193'] == '1', 'Adj_Order'] = 'Adj-N'
    data.loc[data['GB193'] == '2', 'Adj_Order'] = 'N-Adj'
    
    # Filter to SV languages with all features
    sv_langs = data[(data['SV_Order'] == 'SV') & 
                    (data['GEN_Order'] != 'Unknown') & 
                    (data['Adj_Order'] != 'Unknown')].copy()
    
    print(f"\nTotal SV languages with complete data: {len(sv_langs)}")
    
    # MODEL 1: Simple - SV → GEN-N
    model1_match = sv_langs['GEN_Order'] == 'GEN-N'
    model1_count = sum(model1_match)
    model1_prop, model1_lower, model1_upper = wilson_ci(model1_count, len(sv_langs))
    
    # Binomial test
    binom1 = binomtest(model1_count, n=len(sv_langs), p=0.5, alternative='greater')
    
    # MODEL 2: Complex - SV → (GEN-N OR (N-GEN AND N-Adj))
    model2_match = ((sv_langs['GEN_Order'] == 'GEN-N') | 
                    ((sv_langs['GEN_Order'] == 'N-GEN') & (sv_langs['Adj_Order'] == 'N-Adj')))
    model2_count = sum(model2_match)
    model2_prop, model2_lower, model2_upper = wilson_ci(model2_count, len(sv_langs))
    
    # Binomial test
    binom2 = binomtest(model2_count, n=len(sv_langs), p=0.5, alternative='greater')
    
    # Improvement
    improvement = model2_prop - model1_prop
    
    # Cohen's d
    d = cohens_d(model2_prop, model1_prop, len(sv_langs), len(sv_langs))
    
    # Breakdown of N-GEN cases
    n_gen_langs = sv_langs[sv_langs['GEN_Order'] == 'N-GEN']
    n_gen_total = len(n_gen_langs)
    n_gen_with_n_adj = sum(n_gen_langs['Adj_Order'] == 'N-Adj')
    n_gen_with_adj_n = sum(n_gen_langs['Adj_Order'] == 'Adj-N')
    
    n_raising_prop, n_raising_lower, n_raising_upper = wilson_ci(n_gen_with_n_adj, n_gen_total)
    
    # Binomial test for N-raising
    if n_gen_total > 0:
        binom_n_raising = binomtest(n_gen_with_n_adj, n=n_gen_total, p=0.5, alternative='greater')
    else:
        binom_n_raising = None
    
    # Print results
    print("\n" + "-"*70)
    print("MODEL 1: SV → GEN-N (Simple)")
    print("-"*70)
    print(f"Matches: {model1_count}/{len(sv_langs)}")
    print(f"Proportion: {model1_prop:.1%}")
    print(f"95% CI: [{model1_lower:.1%}, {model1_upper:.1%}]")
    print(f"Binomial test p-value: {binom1.pvalue:.6f}")
    print(f"Result: {'✓ Significantly > 50%' if binom1.pvalue < 0.05 else '✗ Not significant'}")
    
    print("\n" + "-"*70)
    print("MODEL 2: SV → (GEN-N OR (N-GEN AND N-Adj)) (Complex)")
    print("-"*70)
    print(f"Matches: {model2_count}/{len(sv_langs)}")
    print(f"Proportion: {model2_prop:.1%}")
    print(f"95% CI: [{model2_lower:.1%}, {model2_upper:.1%}]")
    print(f"Binomial test p-value: {binom2.pvalue:.6f}")
    print(f"Result: {'✓ Significantly > 50%' if binom2.pvalue < 0.05 else '✗ Not significant'}")
    
    print("\n" + "-"*70)
    print("IMPROVEMENT")
    print("-"*70)
    print(f"Absolute: {improvement:.1%} ({improvement*100:.1f} percentage points)")
    print(f"Relative: {(improvement/model1_prop)*100:.1f}% improvement over Model 1")
    
    print("\n" + "-"*70)
    print("EFFECT SIZE")
    print("-"*70)
    print(f"Cohen's d: {d:.3f}")
    
    if d < 0.2:
        effect_interp = "negligible/small"
    elif d < 0.5:
        effect_interp = "small to medium"
    elif d < 0.8:
        effect_interp = "medium to large"
    else:
        effect_interp = "large"
    print(f"Interpretation: {effect_interp} effect")
    
    print("\n" + "-"*70)
    print("N-RAISING HYPOTHESIS TEST")
    print("-"*70)
    print(f"SV languages with N-GEN: {n_gen_total}")
    print(f"  With N-Adj (N-raising pattern): {n_gen_with_n_adj} ({n_raising_prop:.1%})")
    print(f"  With Adj-N (unexpected): {n_gen_with_adj_n} ({n_gen_with_adj_n/n_gen_total:.1%})")
    print(f"95% CI for N-raising: [{n_raising_lower:.1%}, {n_raising_upper:.1%}]")
    
    if binom_n_raising:
        print(f"Binomial test p-value: {binom_n_raising.pvalue:.6f}")
        if n_raising_lower > 0.5:
            print(f"Result: ✓ N-raising hypothesis STRONGLY SUPPORTED (CI lower > 50%)")
        elif n_raising_prop > 0.5 and binom_n_raising.pvalue < 0.05:
            print(f"Result: ✓ N-raising hypothesis SUPPORTED (mean > 50%, p < 0.05)")
        elif n_raising_prop > 0.5:
            print(f"Result: ⚠ N-raising hypothesis WEAKLY SUPPORTED (mean > 50%, but not significant)")
        else:
            print(f"Result: ✗ N-raising hypothesis NOT SUPPORTED")
    
    # Detailed breakdown table
    print("\n" + "-"*70)
    print("DETAILED BREAKDOWN")
    print("-"*70)
    
    # Create contingency table
    breakdown = pd.crosstab(sv_langs['GEN_Order'], sv_langs['Adj_Order'], margins=True)
    print("\nSV Languages: GEN Order × Adjective Order")
    print(breakdown)
    
    # Calculate percentages
    breakdown_pct = pd.crosstab(sv_langs['GEN_Order'], sv_langs['Adj_Order'], 
                                 normalize='index') * 100
    print("\nPercentages (row %):")
    print(breakdown_pct.round(1))
    
    return {
        'n_sv': len(sv_langs),
        'model1_count': model1_count,
        'model1_prop': model1_prop,
        'model1_ci': (model1_lower, model1_upper),
        'model1_p': binom1.pvalue,
        'model2_count': model2_count,
        'model2_prop': model2_prop,
        'model2_ci': (model2_lower, model2_upper),
        'model2_p': binom2.pvalue,
        'improvement': improvement,
        'cohens_d': d,
        'n_gen_total': n_gen_total,
        'n_gen_n_adj': n_gen_with_n_adj,
        'n_gen_adj_n': n_gen_with_adj_n,
        'n_raising_prop': n_raising_prop,
        'n_raising_ci': (n_raising_lower, n_raising_upper),
        'n_raising_p': binom_n_raising.pvalue if binom_n_raising else np.nan,
        'breakdown': breakdown,
        'sv_langs': sv_langs
    }


def analyze_head_marking(data):
    """Same analysis for head-marking languages"""
    
    print("\n" + "="*70)
    print("ANALYZING HEAD-MARKING LANGUAGES")
    print("="*70)
    
    if 'GB431' not in data.columns or 'GB433' not in data.columns:
        print("Error: Head-marking columns not found")
        return None
    
    head_marking = data[(data['GB431'] == '1') | (data['GB433'] == '1')].copy()
    print(f"Total head-marking languages: {len(head_marking)}")
    
    return analyze_full_dataset(head_marking)


def visualize_results(all_results, hm_results=None):
    """Create visualizations comparing models"""
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # 1. Model comparison with CI
    ax = axes[0, 0]
    x = [0, 1]
    
    # All languages
    means_all = [all_results['model1_prop'] * 100, all_results['model2_prop'] * 100]
    ci_lower_all = [all_results['model1_ci'][0] * 100, all_results['model2_ci'][0] * 100]
    ci_upper_all = [all_results['model1_ci'][1] * 100, all_results['model2_ci'][1] * 100]
    errors_all = [[means_all[i] - ci_lower_all[i] for i in range(2)],
                  [ci_upper_all[i] - means_all[i] for i in range(2)]]
    
    bars = ax.bar(x, means_all, color=['lightblue', 'lightcoral'], alpha=0.7, width=0.6)
    ax.errorbar(x, means_all, yerr=errors_all, fmt='none', color='black', 
               capsize=10, linewidth=2, label='95% CI')
    
    # Add values
    for i, (bar, mean, lower, upper) in enumerate(zip(bars, means_all, ci_lower_all, ci_upper_all)):
        ax.text(bar.get_x() + bar.get_width()/2., mean + 3,
               f'{mean:.1f}%\n[{lower:.1f}, {upper:.1f}]',
               ha='center', va='bottom', fontsize=9, fontweight='bold')
    
    ax.axhline(50, color='red', linestyle=':', linewidth=2, alpha=0.5, label='Chance')
    ax.set_ylabel('Proportion (%)', fontsize=12)
    ax.set_title('Model Comparison - All Languages\n(with 95% Wilson CI)', 
                 fontsize=13, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(['Model 1:\nSV→GEN-N', 'Model 2:\nSV→(GEN-N OR\nN-GEN+N-Adj)'])
    ax.legend(loc='lower right')
    ax.set_ylim(0, 105)
    
    # 2. Improvement bar
    ax = axes[0, 1]
    improvement = all_results['improvement'] * 100
    
    bar = ax.bar([0], [improvement], color='green', alpha=0.7, width=0.5)
    ax.text(0, improvement + 0.5, f'{improvement:.1f} pp',
           ha='center', va='bottom', fontsize=11, fontweight='bold')
    ax.axhline(0, color='black', linestyle='-', linewidth=1)
    ax.set_ylabel('Improvement (percentage points)', fontsize=12)
    ax.set_title('Improvement from Adding Adjective Order', fontsize=13, fontweight='bold')
    ax.set_xticks([0])
    ax.set_xticklabels(['Model 2\nvs\nModel 1'])
    ax.set_ylim(min(0, improvement) - 1, improvement + 3)
    
    # 3. Cohen's d with interpretation
    ax = axes[1, 0]
    d = all_results['cohens_d']
    
    bar = ax.bar([0], [d], color='purple', alpha=0.7, width=0.5)
    ax.text(0, d + 0.05, f"d = {d:.3f}",
           ha='center', va='bottom', fontsize=11, fontweight='bold')
    
    ax.axhline(0.2, color='gray', linestyle=':', alpha=0.5, linewidth=1.5)
    ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5, linewidth=1.5)
    ax.axhline(0.8, color='gray', linestyle='-', alpha=0.5, linewidth=1.5)
    ax.text(0.3, 0.2, 'Small', fontsize=8, alpha=0.6)
    ax.text(0.3, 0.5, 'Medium', fontsize=8, alpha=0.6)
    ax.text(0.3, 0.8, 'Large', fontsize=8, alpha=0.6)
    
    ax.set_ylabel("Cohen's d", fontsize=12)
    ax.set_title('Effect Size', fontsize=13, fontweight='bold')
    ax.set_xticks([0])
    ax.set_xticklabels(['Effect of\nAdjective Order'])
    ax.set_ylim(0, max(1, d + 0.2))
    
    # 4. N-raising pattern with CI
    ax = axes[1, 1]
    n_raising = all_results['n_raising_prop'] * 100
    n_lower = all_results['n_raising_ci'][0] * 100
    n_upper = all_results['n_raising_ci'][1] * 100
    
    bar = ax.bar([0], [n_raising], color='orange', alpha=0.7, width=0.5)
    ax.errorbar([0], [n_raising], yerr=[[n_raising - n_lower], [n_upper - n_raising]],
               fmt='none', color='black', capsize=10, linewidth=2, label='95% CI')
    
    ax.text(0, n_raising + 3, f'{n_raising:.1f}%\n[{n_lower:.1f}, {n_upper:.1f}]',
           ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    ax.axhline(50, color='red', linestyle=':', linewidth=2, alpha=0.7, label='Chance (50%)')
    ax.set_ylabel('Percentage', fontsize=12)
    ax.set_title('N-raising Pattern\n(% of SV+N-GEN with N-Adj)', 
                 fontsize=13, fontweight='bold')
    ax.set_xticks([0])
    ax.set_xticklabels(['SV + N-GEN\nLanguages'])
    ax.legend()
    ax.set_ylim(0, 100)
    
    plt.tight_layout()
    plt.savefig('sv_adjective_effect_full_dataset.png', dpi=300, bbox_inches='tight')
    print("\n✓ Visualization saved to: sv_adjective_effect_full_dataset.png")
    plt.close()
    
    # If head-marking results, create comparison
    if hm_results:
        fig, ax = plt.subplots(1, 1, figsize=(10, 6))
        
        x_pos = [0, 1, 3, 4]
        means = [all_results['model1_prop'] * 100, all_results['model2_prop'] * 100,
                hm_results['model1_prop'] * 100, hm_results['model2_prop'] * 100]
        
        ci_lower = [all_results['model1_ci'][0] * 100, all_results['model2_ci'][0] * 100,
                   hm_results['model1_ci'][0] * 100, hm_results['model2_ci'][0] * 100]
        ci_upper = [all_results['model1_ci'][1] * 100, all_results['model2_ci'][1] * 100,
                   hm_results['model1_ci'][1] * 100, hm_results['model2_ci'][1] * 100]
        
        errors = [[means[i] - ci_lower[i] for i in range(4)],
                 [ci_upper[i] - means[i] for i in range(4)]]
        
        colors = ['lightblue', 'lightcoral', 'skyblue', 'salmon']
        bars = ax.bar(x_pos, means, color=colors, alpha=0.7, width=0.7)
        ax.errorbar(x_pos, means, yerr=errors, fmt='none', color='black',
                   capsize=8, linewidth=1.5)
        
        for bar, mean in zip(bars, means):
            ax.text(bar.get_x() + bar.get_width()/2., mean + 2,
                   f'{mean:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
        
        ax.axhline(50, color='red', linestyle=':', linewidth=2, alpha=0.5)
        ax.set_ylabel('Proportion (%)', fontsize=12)
        ax.set_title('All Languages vs Head-Marking Comparison', fontsize=14, fontweight='bold')
        ax.set_xticks(x_pos)
        ax.set_xticklabels(['Model 1\n(All)', 'Model 2\n(All)', 
                           'Model 1\n(Head-Mrk)', 'Model 2\n(Head-Mrk)'])
        ax.set_ylim(0, 100)
        
        plt.tight_layout()
        plt.savefig('sv_adjective_comparison_full_dataset.png', dpi=300, bbox_inches='tight')
        print("✓ Comparison visualization saved to: sv_adjective_comparison_full_dataset.png")
        plt.close()


def main():
    # Load data
    print("Loading data...")
    try:
        grambank = pd.read_csv('grambank_sane_format.csv')
    except FileNotFoundError:
        print("Error: grambank_sane_format.csv not found")
        return
    
    # Convert columns
    required_columns = ['GB130', 'GB065', 'GB193', 'GB431', 'GB433']
    for col in required_columns:
        if col in grambank.columns:
            grambank[col] = grambank[col].astype(str)
    
    print(f"Data loaded: {len(grambank)} languages")
    
    # Analyze full dataset
    print("\n" + "="*70)
    print("TESTING N-RAISING HYPOTHESIS - FULL DATASET ANALYSIS")
    print("="*70)
    print("\nTheoretical Hypothesis:")
    print("  SV languages with N-GEN (unexpected) result from N-raising,")
    print("  which should also produce N-Adj order")
    
    all_results = analyze_full_dataset(grambank)
    
    # Analyze head-marking
    hm_results = analyze_head_marking(grambank)
    
    # Create visualizations
    print("\n" + "="*70)
    print("CREATING VISUALIZATIONS")
    print("="*70)
    visualize_results(all_results, hm_results)
    
    # Save detailed breakdown
    if 'sv_langs' in all_results:
        sv_data = all_results['sv_langs'][['Language', 'SV_Order', 'GEN_Order', 'Adj_Order']]
        sv_data.to_csv('sv_languages_full_breakdown.csv', index=False)
        print("✓ Detailed breakdown saved to: sv_languages_full_breakdown.csv")
    
    print("\n" + "="*70)
    print("ANALYSIS COMPLETE!")
    print("="*70)


if __name__ == "__main__":
    main()

Loading data...
Data loaded: 2467 languages

TESTING N-RAISING HYPOTHESIS - FULL DATASET ANALYSIS

Theoretical Hypothesis:
  SV languages with N-GEN (unexpected) result from N-raising,
  which should also produce N-Adj order

ANALYZING FULL DATASET

Total SV languages with complete data: 1167

----------------------------------------------------------------------
MODEL 1: SV → GEN-N (Simple)
----------------------------------------------------------------------
Matches: 695/1167
Proportion: 59.6%
95% CI: [56.7%, 62.3%]
Binomial test p-value: 0.000000
Result: ✓ Significantly > 50%

----------------------------------------------------------------------
MODEL 2: SV → (GEN-N OR (N-GEN AND N-Adj)) (Complex)
----------------------------------------------------------------------
Matches: 1114/1167
Proportion: 95.5%
95% CI: [94.1%, 96.5%]
Binomial test p-value: 0.000000
Result: ✓ Significantly > 50%

----------------------------------------------------------------------
IMPROVEMENT
-----------